In [1]:
### Analyze MOFA MODEL Results
### Associate the generated factors with sample meta-data covariates and plot the top features per factor

#############################################
# Prerequisites - Load Libraries

In [1]:
source('MS1_Functions.r')

In [3]:
### Inform about start of execution

popup_function_pos('04_Downstream_Factor_Analysis: Execution Started')

In [2]:
source('MS0_Libraries.r')

[1] "/ictstr01/home/icb/bhavishya.nelakuditi/miniconda3/envs/my_jupyter_env/lib/R/library/"


Warning message:
“package ‘ggplot2’ was built under R version 4.3.3”
Warning message:
“package ‘tibble’ was built under R version 4.3.3”
Warning message:
“package ‘purrr’ was built under R version 4.3.3”
Warning message:
“package ‘stringr’ was built under R version 4.3.3”
Warning message:
“package ‘forcats’ was built under R version 4.3.3”
Warning message:
“package ‘lubridate’ was built under R version 4.3.3”
── Attaching core tidyverse packages ──────────────────────────────────────────────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ lubridate 1.9.3     ✔ tibble    3.2.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Warning message:
“packa

In [3]:
source('MS2_Plot_Config.r')

Warning message:
“The `size` argument of `element_line()` is deprecated as of ggplot2 3.4.0.
ℹ Please use the `linewidth` argument instead.”


###############################################
# Preqrequisites Configurations & Parameters

In [6]:
### Load the parameters that are set via the configuration files

In [7]:
### Load configurations file
global_configs = read.csv('configurations/Data_Configs.csv', sep = ',')

In [8]:
head(global_configs,2)

,parameter,value
,<chr>,<chr>
1,data_path,/ictstr01/home/icb/bhavishya.nelakuditi/mofa_workflow/input_data/
2,result_path,/ictstr01/home/icb/bhavishya.nelakuditi/mofa_workflow/results/


In [9]:
data_path = global_configs$value[global_configs$parameter == 'data_path']

In [10]:
data_path

[1] "/ictstr01/home/icb/bhavishya.nelakuditi/mofa_workflow/input_data/"

In [11]:
result_path = global_configs$value[global_configs$parameter == 'result_path']

In [12]:
result_path

[1] "/ictstr01/home/icb/bhavishya.nelakuditi/mofa_workflow/results/"

In [13]:
## Downstream Analysis Configurations (mainly specifying the covariates that should be analyzed in relation to the MOFA factors)

In [14]:
factor_configs = read.csv( 'configurations/04_Factor_Analysis_Configs.csv', sep = ',')
factor_configs = factor_configs[factor_configs$configuration_name != '',]

In [15]:
head(factor_configs,2)

,configuration_name,mofa_result_name,relevant_factors,numeric_covariates,categorical_covariates,top_variable_thres
,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>
1,CGS_v3,CGS_v3_MOFA,"Factor1,Factor2,Factor3,Factor4,Factor5,Factor6","LVEF_discharge,LVEF_admission,catecholamines_days,CRP,Leukocyte,Troponin.T,Plt.Count,IL.6,Lactate,NTproBNP,CK,CK.MB,Crea","risk_with_three_levels,LVEF_outcome,catecholamine_risk,catecholamines,mortality,organ_failure,organ_support,Etiology,death_overall,death_six_months,death_six_year,Condition,Time_point,TP1_vs_HF,TP2_vs_HF,TP3_vs_HF,SCAI,Outcome_comparison_IV,Outcome_comparison_IV_without_HF,Outcome_comparison_V,Outcome_comparison_V_without_HF",0.025


In [16]:
## Get Type color codes from previous script

In [17]:
type_color_codes = read.csv( 'configurations/03_Type_Color_Codes.csv', sep = ',')

In [18]:
head(type_color_codes,2)

,X,color_code
,<int>,<chr>
1,1,#FF7F50
2,2,#D95F02


In [19]:
type_color_codes$color_code

[1] "#FF7F50" "#D95F02" "#377EB8"

# Load Data 

## MOFA Input

In [20]:
#### Load the data based on which the MOFA model was generated (from step 02)

In [21]:
data_list = list()

In [22]:
for(i in 1:nrow(factor_configs)){
    # get data name from configuration
    data_name = factor_configs$configuration_name[i]
    
    # Load data
    path = paste0(result_path, '/02_results/02_Combined_Data_',data_name,'_INTEGRATED.csv')
    if(file.exists(path)){
        data_long = read.csv(path)
        data_long$X = NULL
        print(file.info(path)$mtime)
        print(path)

        # save to list
        data_list[[i]] = data_long
        popup_function_pos(paste0('Loaded: ', '02_Combined_Data_',data_name,'_INTEGRATED.csv'))
           }
    else{popup_function_neg(paste0('Error: Cannot Load: ', '02_Combined_Data_',data_name,'_INTEGRATED.csv', ' Please check whether the previous step has been executed successfully'))}
        
    }

[1] "2024-09-23 18:39:28 CEST"
[1] "/ictstr01/home/icb/bhavishya.nelakuditi/mofa_workflow/results//02_results/02_Combined_Data_CGS_v3_INTEGRATED.csv"


In [23]:
## Example of loaded data

In [24]:
head(data_long,2)

,sample_id,variable,value,type,gene
,<chr>,<chr>,<dbl>,<chr>,<chr>
1,HF10,Activated.T.cells__AAK1,0.4022501,Activated.T.cells,AAK1
2,HF11,Activated.T.cells__AAK1,0.0000000,Activated.T.cells,AAK1


## MOFA Model

### Load estimated model

In [25]:
### Load the MOFA model(s) that should be analyzed

In [26]:
model_list = list()

In [27]:
for(i in 1:nrow(factor_configs)){
    # get data name from configuration
    mofa_name = factor_configs$mofa_result_name[i]
 
    # Load model
    model_name =  paste0("03_MOFA_MODEL_",mofa_name, '.hdf5')
    outfile = file.path( paste0(result_path, '/03_results/',  model_name) )
    
    if(file.exists(outfile)){
        model = load_model(outfile, verbose = TRUE)

        print(file.info(outfile)$mtime)
        print(outfile)

        # save to list
        model_list[[i]] = model
        popup_function_pos(paste0('Loaded: ', model_name))
        }
    else {  popup_function_neg(paste0('Error: ',result_path, '/03_results/ ', model_name, 'could not be loaded. Check whether the previous steps have been executed successfully'))}
    }
    

Loading data...

Loading expectations for 2 nodes...

Loading model options...

Loading training options and statistics...

Assigning names to the different dimensions...

Re-ordering factors by their variance explained...

Doing quality control...

Checking views names...

Checking groups names...

Checking samples names...

Checking features names...

Checking dimensions...

Checking there are no features with complete missing values...

Checking sample covariates...

Checking expectations...

Checking for intercept factors...

Checking for highly correlated factors...



[1] "2024-09-23 18:42:02 CEST"
[1] "/ictstr01/home/icb/bhavishya.nelakuditi/mofa_workflow/results//03_results/03_MOFA_MODEL_CGS_v3_MOFA.hdf5"


### Factor Data

In [28]:
### Load the factor data to the corresponding MOFA model

In [29]:
factor_data_list = list()

In [30]:
for(i in 1:nrow(factor_configs)){
    # get data name from configuration
    mofa_name = factor_configs$mofa_result_name[i]
    
    # load data
    path = paste0(result_path, '/03_results/', '03_Factor_Data_', mofa_name, '.csv')
    
    if(file.exists(path)){
        factors = read.csv(path)
        print(file.info(path)$mtime)
        print(path)

        # Save to list
        factor_data_list[[i]] = factors
        popup_function_pos(paste0('Loaded: ','03_Factor_Data_', mofa_name, '.csv'))
        }
    else{ popup_function_neg(paste0('Error: ','03_Factor_Data_', mofa_name, '.csv', ' could not be loaded. Check whether the previous steps have been executed successfully'))} 
        
        
        
    }
    

[1] "2024-09-23 18:42:04 CEST"
[1] "/ictstr01/home/icb/bhavishya.nelakuditi/mofa_workflow/results//03_results/03_Factor_Data_CGS_v3_MOFA.csv"


In [31]:
head(factors,2)

,Factor1,Factor2,Factor3,Factor4,Factor5,Factor6,Factor7,Factor8,Factor9,Factor10,Factor11,Factor12,Factor13,sample_id
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
1,0.1553547,0.01219151,1.373604,-0.9342654,-0.08166236,0.09966231,-0.738721896,-1.4202759,0.1809295,-0.6330105,-0.33272946,0.7804502,0.6657295,HF1
2,-1.3159462,0.98926091,1.290839,0.2897947,0.50661458,-0.50892308,0.007523418,-0.2540576,0.4483722,-0.2281557,-0.07792966,-0.2514229,-0.1821789,HF10


### Weight Data

In [32]:
# Load the feature factor weights to the corresponding MOFA model

In [33]:
weight_data_list = list()

In [34]:
for(i in 1:nrow(factor_configs)){
    # get data name from configuration
    mofa_name = factor_configs$mofa_result_name[i]
    
    # laod data
    path = paste0(result_path, '/03_results/', '03_Weight_Data_',  mofa_name, '.csv')
    
    if(file.exists(path)){
        weight_data = read.csv(path)

        print(file.info(path)$mtime)
        print(path)

        # save to list
        weight_data_list[[i]] = weight_data
        popup_function_pos(paste0('Loaded: ','03_Weight_Data_',  mofa_name, '.csv'))}
    
    else{ popup_function_neg(paste0('Error: ','03_Weight_Data_',  mofa_name, '.csv', ' could not be loaded. Check whether the previous steps have been executed successfully and the file exists.'))} 
    
    }
    
    

[1] "2024-09-23 18:42:04 CEST"
[1] "/ictstr01/home/icb/bhavishya.nelakuditi/mofa_workflow/results//03_results/03_Weight_Data_CGS_v3_MOFA.csv"


## Sample Meta Data

In [35]:
## Load the sample meta-data file that contains covariates that should be associated to factor data

In [36]:
file_path = paste0(data_path, 'Prepared_Sample_Meta_Data', '.csv')
if(file.exists(file_path)){
    sample_data = read.csv(paste0(data_path, 'Prepared_Sample_Meta_Data', '.csv'))
    sample_data$X = NULL
    popup_function_pos(paste0('Loaded: ','Prepared_Sample_Meta_Data', '.csv'))} else{ popup_function_neg(paste0('Error: ','Prepared_Sample_Meta_Data', '.csv', ' could not be loaded. Please add the file to the specified input data folder.'))}

In [37]:
### Check whether required columns are in loaded data

In [38]:
if(sum(strsplit(factor_configs$numeric_covariates , ',')[[1]] %in% colnames(sample_data)) != length(strsplit(factor_configs$numeric_covariates , ',')[[1]])){
     popup_function_neg(paste0('Error: ', strsplit(factor_configs$numeric_covariates , ',')[[1]][!strsplit(factor_configs$numeric_covariates , ',')[[1]] %in% colnames(sample_data)], ' is not included as column within the specified sample dataset.'))}
    

In [39]:
if(sum(strsplit(factor_configs$categorical_covariates , ',')[[1]] %in% colnames(sample_data)) != length(strsplit(factor_configs$categorical_covariates , ',')[[1]])){
     popup_function_neg(paste0('Error: ', strsplit(factor_configs$categorical_covariates , ',')[[1]][!strsplit(factor_configs$categorical_covariates , ',')[[1]] %in% colnames(sample_data)], ' is not included as column within the specified sample dataset.'))}

In [40]:
if(! 'sample_id' %in% colnames(sample_data)){
    popup_function_neg(paste0('Error: ', 'columns sample_id is not included in sample data and is required'))}

# Downstream Analysis of generated model

# Extract and prepare data for plots

## Merge factors and sample data

In [41]:
## Combine the factor and sample data table

In [42]:
factor_data_processed = list()

In [43]:
for( i in 1:length(factor_data_list)){
    
    ## Transform data to long format
    factors_long = melt(factor_data_list[[i]], id.vars = 'sample_id')
    
    ## Add sample data info
    merged_data_long = merge(factors_long, sample_data, by.x = 'sample_id', by.y = 'sample_id')
    
     ### Filter on relevant factors
     relevant_factors = unlist(str_split(factor_configs$relevant_factors[i], ','))  # get relevant factor from configuration data
     merged_data_long = merged_data_long[merged_data_long$variable %in% relevant_factors,]
    
     factor_data_processed[[i]] = merged_data_long
    }

In [44]:
head(factor_data_processed[[i]],2)

,sample_id,variable,value,risk,risk_with_three_levels,LVEF_discharge,LVEF_outcome,catecholamine_risk,lactate_clearance,LVEF_admission,⋯,CRP,Leukocyte,Troponin.T,Plt.Count,IL.6,Lactate,NTproBNP,CK,CK.MB,Crea
,<chr>,<fct>,<dbl>,<chr>,<chr>,<int>,<chr>,<chr>,<chr>,<int>,⋯,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<int>,<int>,<int>,<dbl>
1,HF1,Factor1,0.15535473,,,NA,,,,NA,⋯,0.1,10.9,0.031,287,NA,NA,NA,80,21,0.8
6,HF1,Factor2,0.01219151,,,NA,,,,NA,⋯,0.1,10.9,0.031,287,NA,NA,NA,80,21,0.8


## Extract explained variance for plotting

In [45]:
## Get the explained variance information from the MOFA model (these values will be plotted later)

In [46]:
explained_variance = lapply(model_list, function(x){
    
    # extract variance per factor from model
    data = x@cache$variance_explained$r2_per_factor[[1]]
    
    # extract total variance from model
    total_variance = data.frame( view = rownames(x@cache[["variance_explained"]]$r2_total$group1,2),
                             total_variance = x@cache[["variance_explained"]]$r2_total$group1)
    
    # extract total variance per factor
    total_variance_factor = data.frame(factor = names(rowMeans(x@cache$variance_explained$r2_per_factor[[1]])),
                                   mean_variance = rowMeans(x@cache$variance_explained$r2_per_factor[[1]]))
    
    data = melt(data)
    # merge different variance values
    data = merge(data, total_variance, by.x = 'Var2', by.y = 'view')
    
    })

In [47]:
head(explained_variance[[1]],2)

,Var2,Var1,value,total_variance
,<fct>,<fct>,<dbl>,<dbl>
1,Activated.T.cells,Factor1,15.858381,47.93013
2,Activated.T.cells,Factor2,7.702808,47.93013


## Prepare weight data

In [48]:
### Adjust the format of the feature factor weight data to long

In [49]:
feature_weights_list = lapply(weight_data_list, function(x){
    feature_weights_long = melt(x, id.vars = c('variable_name', 'type'))
    
    # adjust formatting of columns
    feature_weights_long$view = feature_weights_long$type
    feature_weights_long$gene = str_replace(feature_weights_long$variable_name, '.*__', '')
    
    return(feature_weights_long)
    })
    

In [50]:
head(feature_weights_list[[1]],2)

,variable_name,type,variable,value,view,gene
,<chr>,<chr>,<fct>,<dbl>,<chr>,<chr>
1,Activated.T.cells__AAK1,Activated.T.cells,Factor1,0.4170012,Activated.T.cells,AAK1
2,Activated.T.cells__ABLIM1,Activated.T.cells,Factor1,-0.1381327,Activated.T.cells,ABLIM1


## Get top features per factor and amounts for diff thresholds

In [51]:
## Get the x% of top features per factor based on the specified threshold (x) in the configuration file

In [52]:
geneset_oi_amounts_list = list()

In [53]:
for(i in 1:length(feature_weights_list)){
    
    feature_weights_long = feature_weights_list[[i]]
    
    # get threshold for top variables from configuration
    top_variable_perc = factor_configs$top_variable_thres[i]
    
    
    ## Define amount of top genes per fraction 
    geneset_oi_pos_per_factor_analyze = feature_weights_long %>% group_by(variable) %>% dplyr::arrange( desc(value),  .by_group = TRUE)  %>% top_frac(  as.numeric(top_variable_perc), value)
    geneset_oi_pos_per_factor_analyze$direction = 'positive'
    
    geneset_oi_neg_per_factor_analyze = feature_weights_long %>% group_by(variable) %>% dplyr::arrange(desc(value),  .by_group = TRUE)  %>% top_frac( - as.numeric(top_variable_perc), value)
    geneset_oi_neg_per_factor_analyze$direction = 'negative'
    
    geneset_oi_analyze = rbind(geneset_oi_pos_per_factor_analyze, geneset_oi_neg_per_factor_analyze)
    geneset_oi_analyze$fraction =  as.numeric(top_variable_perc)

    
    ## Calculate the amount of top features per type
    dimensions = unique(feature_weights_long[,c('view', 'variable')])
    
    amount_geneset_oi_type = geneset_oi_analyze %>% group_by(type, view, variable) %>% dplyr::count()
    amount_geneset_oi_type = merge(dimensions, amount_geneset_oi_type, all.x = TRUE) # to avoid missing dimensions
    amount_geneset_oi_type$fraction = as.numeric(top_variable_perc)
    
    geneset_oi_amounts = amount_geneset_oi_type
    
    ## Calculate the total amount of features per type
    features_per_type = feature_weights_long %>% group_by(type, view, variable) %>% dplyr::count()
    colnames(features_per_type) = c('type', 'view', 'variable', 'amount_features')
    
    ## Merge and calculate percentage
    geneset_oi_amounts = merge(  geneset_oi_amounts,features_per_type, all.x = TRUE)
    geneset_oi_amounts$percentage = geneset_oi_amounts$n / geneset_oi_amounts$amount_features
    
    ## Adjust NA's
    geneset_oi_amounts[is.na(geneset_oi_amounts)] = 0 # NA when there are no features for this dimension among top %
    
    ## save for the threshold/ config
    geneset_oi_amounts_list[[i]] = geneset_oi_amounts
    }
    
    
    
    

In [54]:
head(    geneset_oi_amounts_list[[i]],2)

,view,variable,type,n,fraction,amount_features,percentage
,<chr>,<fct>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
1,Activated.T.cells,Factor1,Activated.T.cells,89,0.025,886,0.10045147
2,Activated.T.cells,Factor2,Activated.T.cells,53,0.025,886,0.05981941


# Plots

In [55]:
### Generate the plots to evaluate the factor values in relation to different sample-meta data covariates

## Investigate relationship of factors with numeric values

In [56]:
### Calculate correlatins with numeric features

In [57]:
head(factor_data_processed[[1]],2)

,sample_id,variable,value,risk,risk_with_three_levels,LVEF_discharge,LVEF_outcome,catecholamine_risk,lactate_clearance,LVEF_admission,⋯,CRP,Leukocyte,Troponin.T,Plt.Count,IL.6,Lactate,NTproBNP,CK,CK.MB,Crea
,<chr>,<fct>,<dbl>,<chr>,<chr>,<int>,<chr>,<chr>,<chr>,<int>,⋯,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<int>,<int>,<int>,<dbl>
1,HF1,Factor1,0.15535473,,,NA,,,,NA,⋯,0.1,10.9,0.031,287,NA,NA,NA,80,21,0.8
6,HF1,Factor2,0.01219151,,,NA,,,,NA,⋯,0.1,10.9,0.031,287,NA,NA,NA,80,21,0.8


In [58]:
figure_name = "FIG04_Factor_Association_with_numeric_features_"  # Define the name of the plot

In [59]:
# Set the sizes of the plot
width_par = 8.07
height_par = 3.5

In [60]:
for(j in 1:length(factor_data_processed)){
    ## get the variables
    numeric_variables = unlist(str_split(factor_configs$numeric_covariates[j], ','))
    
    
    ## calculate correlations
    cor_plot = list()
    
    ## get the data
    merged_data_long = factor_data_processed[[j]]
    
    ### calculate correlations
    for(i in numeric_variables){
        cor_plot[[i]] = ggplot(merged_data_long, aes(x = value, y = get(i))) + facet_wrap(.~ variable, scale = 'free') +
        geom_point(size = 0.2) + plot_config + geom_smooth(method='lm', col = 'blue3', se = FALSE) + stat_cor(method = 'pearson') + ylab(i)


        }
    
    
    ### Plot the scatterplots
    
    # get the name for saving
    mofa_name = factor_configs$mofa_result_name[j]
    
    pdf(paste0('figures/04_figures/', figure_name,mofa_name, '.pdf'), width =width_par, height =height_par, onefile = TRUE)
    for (i in names(cor_plot)) {
      print(cor_plot[[i]])  
    }
    dev.off()
    }
    
    

`geom_smooth()` using formula = 'y ~ x'
Warning message:
“Removed 420 rows containing non-finite outside the scale range (`stat_smooth()`).”
Warning message:
“Removed 420 rows containing non-finite outside the scale range (`stat_cor()`).”
Warning message:
“Removed 420 rows containing missing values or values outside the scale range (`geom_point()`).”
`geom_smooth()` using formula = 'y ~ x'
Warning message:
“Removed 390 rows containing non-finite outside the scale range (`stat_smooth()`).”
Warning message:
“Removed 390 rows containing non-finite outside the scale range (`stat_cor()`).”
Warning message:
“Removed 390 rows containing missing values or values outside the scale range (`geom_point()`).”
`geom_smooth()` using formula = 'y ~ x'
Warning message:
“Removed 420 rows containing non-finite outside the scale range (`stat_smooth()`).”
Warning message:
“Removed 420 rows containing non-finite outside the scale range (`stat_cor()`).”
Warning message:
“Removed 420 rows containing missing v

## Investigate relationship with categorical values

In [61]:
### Generate boxplots to evaluate the factor value difference for categorical sample meta data variables

In [88]:
# Specific Text Descriptions:
xlabel = xlab('') 
ylabel = ylab('Factor Value')

In [89]:
# Specify Figure Name
figure_name = paste0("FIG04_Factor_Association_with_categorical_features_")

In [90]:
# Specify sizes of the plot
width_par = 8.07
height_par = 5

In [91]:
head(merged_data_long$Condition)

[1] "heart_failure" "heart_failure" "heart_failure" "heart_failure"
[5] "heart_failure" "heart_failure"

In [99]:
custom_colors <- list(
    "Condition" = scale_color_manual(values = c("heart_failure" = "black", "cardiogenic_shock" = "magenta")),
    "risk_with_three_levels" = scale_color_manual(values = c("low_risk_3" = "darkgreen", "high_risk_3" = "orange")),
    "default" = scale_color_discrete()
)

for(j in 1:length(factor_data_processed)){
    ## get the variables
    categorical_variables = unlist(str_split(factor_configs$categorical_covariates[j], ','))
    
    ## calculate correlations
    cor_plot = list()
    
    ## get the data
    merged_data_long = factor_data_processed[[j]]
    
    ## Get MOFA name for file naming
    mofa_name = factor_configs$mofa_result_name[j]
    
    # Loop through each categorical variable and create separate PDF
    for(i in categorical_variables){
        variable = i
        merged_data_long$condition = merged_data_long[,variable]
        vis_data = merged_data_long
        faceting_variable <- variable
        vis_data <- vis_data %>%
            filter(!is.na(.data[[faceting_variable]]) & .data[[faceting_variable]] != "")
        
        # Get the number of unique conditions
        unique_values <- unique(vis_data$condition[!is.na(vis_data$condition)])
        unique_conditions <- length(unique_values)
        
        # Choose statistical test based on number of conditions
        if(unique_conditions == 2) {
            # For two conditions, use Wilcoxon rank sum test
            stat_test <- stat_compare_means(method = "wilcox.test", label = "p.format")
        } else if(unique_conditions > 2) {
            # For more than two conditions, use Kruskal-Wallis test
            stat_test <- stat_compare_means(method = "kruskal.test", label = "p.format")
        } else {
            # For one or zero conditions, no test
            stat_test <- NULL
        }
        
        # Set ordering and colors based on values present
        if(all(c("heart_failure", "cardiogenic_shock") %in% unique_values)) {
            vis_data$condition <- factor(vis_data$condition, levels = c("heart_failure", "cardiogenic_shock"))
            color_scale <- scale_color_manual(values = c("heart_failure" = "black", "cardiogenic_shock" = "magenta"))
        } else if(all(c("low_risk_3", "high_risk_3") %in% unique_values)) {
            vis_data$condition <- factor(vis_data$condition, levels = c("low_risk_3", "high_risk_3"))
            color_scale <- scale_color_manual(values = c("low_risk_3" = "darkgreen", "high_risk_3" = "orange"))
        } else {
            # For other variables, use default ordering and colors
            color_scale <- scale_color_discrete()
        }
        
        # Create the plot
        g = ggplot(vis_data[!is.na(vis_data$condition),], aes(x=condition, y=value, col = condition)) + 
            facet_grid(.~variable) +
            plot_config +
            xlabel + 
            ylabel + 
            ggtitle('Pattern of factor values') + 
            theme(legend.position = "bottom", axis.text.x = element_blank()) +
            geom_boxplot(outlier.size = 0.05) + 
            geom_point(size = 0.5) + 
            {if(!is.null(stat_test)) stat_test else NULL} +
            color_scale 
        
        # Save each variable to its own PDF file
        pdf(paste0('figures/04_figures/', figure_name, mofa_name, '_', variable, '.pdf'), 
            width = width_par, height = height_par)
        print(g)
        dev.off()  # Close the PDF device after each plot
    }
}

In [71]:
head(vis_data)

,sample_id,variable,value,risk,risk_with_three_levels,LVEF_discharge,LVEF_outcome,catecholamine_risk,lactate_clearance,LVEF_admission,⋯,Leukocyte,Troponin.T,Plt.Count,IL.6,Lactate,NTproBNP,CK,CK.MB,Crea,condition
,<chr>,<fct>,<dbl>,<chr>,<chr>,<int>,<chr>,<chr>,<chr>,<int>,⋯,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<int>,<int>,<int>,<dbl>,<chr>
1,KS1.1,Factor1,-1.0001036,low_risk,low_risk_3,18,bad_LVEF,,good,20,⋯,13.8,NA,320,15.2,2.9,NA,53,NA,0.6,good
2,KS1.1,Factor4,0.2029972,low_risk,low_risk_3,18,bad_LVEF,,good,20,⋯,13.8,NA,320,15.2,2.9,NA,53,NA,0.6,good
3,KS1.1,Factor3,0.1910931,low_risk,low_risk_3,18,bad_LVEF,,good,20,⋯,13.8,NA,320,15.2,2.9,NA,53,NA,0.6,good
4,KS1.1,Factor6,-0.1627209,low_risk,low_risk_3,18,bad_LVEF,,good,20,⋯,13.8,NA,320,15.2,2.9,NA,53,NA,0.6,good
5,KS1.1,Factor2,1.2014375,low_risk,low_risk_3,18,bad_LVEF,,good,20,⋯,13.8,NA,320,15.2,2.9,NA,53,NA,0.6,good
6,KS1.1,Factor5,-0.4717768,low_risk,low_risk_3,18,bad_LVEF,,good,20,⋯,13.8,NA,320,15.2,2.9,NA,53,NA,0.6,good


## Feature Overview per Factor

In [72]:
## Generate the overview of the amount of top x% of features per factor and view (in relation to total amount of features per view)

In [73]:
## Plot the heatmap showing the explained variance of the factor per view

In [74]:
variance_plots = list()

In [75]:
for(j in 1:nrow(factor_configs)){

    ## get the relevant factor and top variable fraction
    factor_var = unlist(str_split(factor_configs$relevant_factors[j], ','))
    
    
    ## Explained Variance Heatmap Plot (for each factor)
    explained_variance_heatmap = list()
    for(i in factor_var ){
        data = explained_variance[[j]]   # get prepared variance data
        data$Var2 = as.character(data$Var2)
        data$Var2 = factor(data$Var2, levels = sort(unique(data$Var2)))  # recode to ensure right ordering
        
        data_plot = data[data$Var1 == i,]
        data_plot$Var1 = 'Explained'

        explained_variance_heatmap[[i]] = ggplot() + scale_fill_gradient(low="white", high="black") + 
        ylabel + xlabel + plot_config_heatmap +  theme(axis.text.y = element_text(hjust = 0, vjust = 0.5)) +
        geom_tile(data = data_plot, mapping = aes(Var1,  Var2, fill= value)
                 )  +
        ggtitle(i)
        }
    
    variance_plots[[j]] =  explained_variance_heatmap
    }

In [76]:
#variance_plots[[1]]

In [77]:
## Generate the barplots with showing the amount of top x% of features

In [78]:
barplot_top_features_percentages = list()
barplot_top_features_absolute = list()

In [79]:
for(j in 1:nrow(factor_configs)){
    
    # get configurations
    top_var_fraction = factor_configs$top_variable_thres[j]
    factor_var = unlist(str_split(factor_configs$relevant_factors[j], ','))
    
    ## Barplots with top features per factor
    
    ## 1: Percentages (amount of top x% of features in relation to total amount)
    xlabel = xlab('View') 
    ylabel = ylab('Percentage of total features')
    
    percentage_plot_1_perc = list()
    
    for(i in factor_var ){

            geneset_oi_amounts = geneset_oi_amounts_list[[j]]
            geneset_oi_amounts$view  = as.character(geneset_oi_amounts$view)
            geneset_oi_amounts$view = factor(geneset_oi_amounts$view, levels = sort(unique(geneset_oi_amounts$view))) # recode to ensure right ordering

            percentage_plot_1_perc[[i]] = ggplot(data = geneset_oi_amounts[(geneset_oi_amounts$variable == i),], aes(x = view, y = percentage*100, fill = view)) +
            xlabel + 
            ylabel + 
            plot_config + 
            geom_bar(stat="identity") + coord_flip() + theme(legend.position = 'none') +
            ggtitle(paste0('Top ', 2*as.numeric( top_var_fraction) *100, '% of features')) +
            geom_hline(yintercept = 2*as.numeric( top_var_fraction)*100, 
                    color = "black", size=1) + scale_fill_manual(values = type_color_codes$color_code)
        
    }
    barplot_top_features_percentages[[j]] =  percentage_plot_1_perc 
    
    
    
    ## 2: Absolute Values (absolute amount of top x% of features of the view)
    
    xlabel = xlab('View') 
    ylabel = ylab('Amount features')
    
    absolute_plot_1_perc = list()
    
    # one selected threshold + absolute amount
    for(i in factor_var ){
            geneset_oi_amounts = geneset_oi_amounts_list[[j]]
            geneset_oi_amounts$view = as.character(geneset_oi_amounts$view)
            geneset_oi_amounts$view = factor(geneset_oi_amounts$view, levels = sort(unique(geneset_oi_amounts$view)))# recode to ensure right ordering

            absolute_plot_1_perc[[i]] = ggplot(data = geneset_oi_amounts[(geneset_oi_amounts$variable == i),], aes(x = view, y = n, fill = view)) +
            xlabel + 
            ylabel + 
            plot_config + 
            geom_bar(stat="identity") + coord_flip()  + theme(legend.position = 'none')+ 
            ggtitle(paste0('Top ', 2*as.numeric( top_var_fraction) *100, '% of features')) + scale_fill_manual(values = type_color_codes$color_code) # TBD: maybe improve even with default value + specifying all colors via table
        }
    
    barplot_top_features_absolute[[j]] = absolute_plot_1_perc
       

}  

Warning message:
“Using `size` aesthetic for lines was deprecated in ggplot2 3.4.0.
ℹ Please use `linewidth` instead.”


In [80]:
#barplot_top_features_percentages[[1]]

In [81]:
## Combine the plots and save

In [82]:
# Specify the figure name
figure_name = paste0( "FIG04_Top_Feature_Overview_per_Factor")

In [83]:
# Specify the sizes of the plot
width_par = 8.07
height_par =2.8

In [ ]:
for(j in 1:nrow(factor_configs)){
    mofa_name = factor_configs$mofa_result_name[j]
    
    # get the relvant plot
    explained_variance_heatmap = variance_plots[[j]]
    absolute_plot_1_perc = barplot_top_features_absolute[[j]]
    percentage_plot_1_perc = barplot_top_features_percentages[[j]]


    pdf(paste0('figures/04_figures/', figure_name, '_',   mofa_name, '.pdf'), width =width_par, height =height_par)
    for( i in 1:length(explained_variance_heatmap)){
        legend = get_legend(explained_variance_heatmap[[i]])

        combined1 = ggarrange(explained_variance_heatmap[[i]] + theme(legend.position = 'none'),
                             absolute_plot_1_perc[[i]] + theme(axis.text.y = element_blank(),axis.ticks.y = element_blank(),axis.title.y = element_blank() ), 
                             percentage_plot_1_perc[[i]] + theme(axis.text.y = element_blank(),axis.ticks.y = element_blank(),axis.title.y = element_blank() ),  
                             nrow=1, widths = c(2.2,1,1))
        combined1 = annotate_figure(combined1, right = legend)

        print( combined1)
        }
    dev.off()   
    }

In [ ]:
### Inform about finalkzation of execution

popup_function_pos('04_Downstream_Factor_Analysis: Execution Finished')

In [ ]:
Sys.sleep(20)
popup_function_info('04_Downstream_Factor_Analysis')

In [ ]:
### correlation of max clinical values to Factor values

In [1]:
max_min = read.csv("CGS_clinical_data_max_min.csv")
head(max_min)

,sample_id,max..CRP,CRP_level,CRP_level_binary,max..IL.6,IL.6_level_binary,IL.6_level,max..leukocyte.count,leukocyte_level_binary,leukocyte_level,⋯,GPT_level,max..GOT,GOT_level_binary,GOT_level,max..Bilirubin,bilirubin_level_binary,bilirubin_level,max..PCT,PCT_level_binary,PCT_level
,<chr>,<dbl>,<lgl>,<chr>,<dbl>,<lgl>,<chr>,<dbl>,<lgl>,<chr>,⋯,<chr>,<int>,<lgl>,<chr>,<dbl>,<lgl>,<chr>,<dbl>,<lgl>,<chr>
1,KS27.1,7.7,FALSE,low_CRP,NA,FALSE,low_IL-6,5.82,FALSE,low_leukocyte,⋯,low_GPT,64,FALSE,low_GOT,0.6,FALSE,low_bilirubin,NA,FALSE,low_PCT
2,KS1.1,27.0,TRUE,high_CRP,309.0,TRUE,high_IL-6,27.70,TRUE,high_leukocyte,⋯,low_GPT,85,FALSE,low_GOT,1.7,FALSE,low_bilirubin,0.6,FALSE,low_PCT
3,KS10.1,23.1,TRUE,high_CRP,892.0,TRUE,high_IL-6,33.60,TRUE,high_leukocyte,⋯,low_GPT,68,FALSE,low_GOT,4.8,TRUE,high_bilirubin,15.5,TRUE,high_PCT
4,KS11.1,23.4,TRUE,high_CRP,13033.0,TRUE,high_IL-6,17.90,FALSE,low_leukocyte,⋯,low_GPT,130,FALSE,low_GOT,10.0,TRUE,high_bilirubin,26.9,TRUE,high_PCT
5,KS12.1,12.9,TRUE,high_CRP,69.4,FALSE,low_IL-6,14.00,FALSE,low_leukocyte,⋯,high_GPT,1651,TRUE,high_GOT,1.4,FALSE,low_bilirubin,0.3,FALSE,low_PCT
6,KS13.1,21.3,TRUE,high_CRP,254.0,TRUE,high_IL-6,32.90,TRUE,high_leukocyte,⋯,high_GPT,727,TRUE,high_GOT,3.7,TRUE,high_bilirubin,0.4,FALSE,low_PCT


In [2]:
factor_data = read.csv("/ictstr01/home/icb/bhavishya.nelakuditi/mofa_workflow/results/03_results/03_Factor_Data_CGS_v3_MOFA.csv")
head(factor_data)

,Factor1,Factor2,Factor3,Factor4,Factor5,Factor6,Factor7,Factor8,Factor9,Factor10,Factor11,Factor12,Factor13,sample_id
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
1,0.1553547,0.01219151,1.3736038,-0.93426545,-0.08166236,0.09966231,-0.738721896,-1.42027586,0.18092948,-0.6330105,-0.332729461,0.78045024,0.66572949,HF1
2,-1.3159462,0.98926091,1.2908387,0.28979471,0.50661458,-0.50892308,0.007523418,-0.25405759,0.44837224,-0.2281557,-0.077929655,-0.25142288,-0.18217892,HF10
3,-0.2156277,-1.06062850,1.2593312,0.49595657,0.02864646,0.37552060,-0.070152401,0.04071692,-0.47304877,0.1357894,-0.004220344,-0.21509484,0.08225105,HF11
4,-1.1068152,0.88894605,1.7965191,-0.15320224,0.33775009,-0.57366225,0.243760460,-0.35551319,-0.07337548,-0.0808971,0.100544362,-0.26291809,0.09855957,HF12
5,-1.6710594,1.02814152,0.9439853,0.16673008,1.36395125,0.12960826,0.455817440,-0.67245306,0.18199052,-0.4125291,-0.229430686,-0.03090595,-0.36484312,HF13
6,-1.2785599,1.65247848,1.1872968,-0.05706409,0.47102526,-0.67478579,-0.036546924,0.11653138,0.05618972,0.1941608,-0.030343686,0.10389498,0.24589247,HF14


In [3]:
merged_data = merge(factor_data, max_min, by = "sample_id")
head(merged_data)

,sample_id,Factor1,Factor2,Factor3,Factor4,Factor5,Factor6,Factor7,Factor8,Factor9,⋯,GPT_level,max..GOT,GOT_level_binary,GOT_level,max..Bilirubin,bilirubin_level_binary,bilirubin_level,max..PCT,PCT_level_binary,PCT_level
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<chr>,<int>,<lgl>,<chr>,<dbl>,<lgl>,<chr>,<dbl>,<lgl>,<chr>
1,KS1.1,-1.0001036,1.2014375,0.19109306,0.2029972,-0.47177681,-0.16272093,-0.28336810,-0.01341458,0.25001705,⋯,low_GPT,85,FALSE,low_GOT,1.7,FALSE,low_bilirubin,0.6,FALSE,low_PCT
2,KS10.1,-0.2717517,-1.5116551,-0.86043833,0.1825573,1.82945215,0.05442225,0.01398519,-0.04458514,-0.05696874,⋯,low_GPT,68,FALSE,low_GOT,4.8,TRUE,high_bilirubin,15.5,TRUE,high_PCT
3,KS11.1,-1.2027951,-0.7874092,-1.00768180,-0.3555820,-0.08853537,-0.28274766,-0.77031785,1.92947208,0.11560659,⋯,low_GPT,130,FALSE,low_GOT,10.0,TRUE,high_bilirubin,26.9,TRUE,high_PCT
4,KS12.1,-1.3254246,0.6048656,0.06634458,-0.7936355,0.38171278,-0.39505495,0.47826480,0.69586578,-0.80711825,⋯,high_GPT,1651,TRUE,high_GOT,1.4,FALSE,low_bilirubin,0.3,FALSE,low_PCT
5,KS13.1,-1.3431659,2.8188848,-0.32873567,-0.5117440,-0.17110475,-0.34592955,-0.77788595,-0.28147827,0.06260042,⋯,high_GPT,727,TRUE,high_GOT,3.7,TRUE,high_bilirubin,0.4,FALSE,low_PCT
6,KS14.1,-1.0655831,0.6177216,-1.34651578,-0.5478794,0.59715424,-0.08439325,0.75623158,-0.23113605,-0.40220035,⋯,high_GPT,1250,TRUE,high_GOT,4.3,TRUE,high_bilirubin,8.0,TRUE,high_PCT


In [4]:
colnames(merged_data)

[1] "sample_id"               "Factor1"                
 [3] "Factor2"                 "Factor3"                
 [5] "Factor4"                 "Factor5"                
 [7] "Factor6"                 "Factor7"                
 [9] "Factor8"                 "Factor9"                
[11] "Factor10"                "Factor11"               
[13] "Factor12"                "Factor13"               
[15] "max..CRP"                "CRP_level"              
[17] "CRP_level_binary"        "max..IL.6"              
[19] "IL.6_level_binary"       "IL.6_level"             
[21] "max..leukocyte.count"    "leukocyte_level_binary" 
[23] "leukocyte_level"         "max..Crea"              
[25] "crea_level_binary"       "crea_level"             
[27] "max..CK"                 "CK_level_binary"        
[29] "CK_level"                "max..CK.MB"             
[31] "CK_MB_level_binary"      "CK_MB_level"            
[33] "max..Trop.T"             "troponin_level_binary"  
[35] "troponin_level"          "max....neutrophils"     
[37] "neutrophil_level_binary" "neutrophil_level"       
[39] "max..lactate"            "lactate_level_binary"   
[41] "lactate_level"           "min..platelet.count"    
[43] "platelet_level_binary"   "platelet_level"         
[45] "max..Urea"               "urea_level_binary"      
[47] "urea_level"              "max..GPT"               
[49] "GPT_level_binary"        "GPT_level"              
[51] "max..GOT"                "GOT_level_binary"       
[53] "GOT_level"               "max..Bilirubin"         
[55] "bilirubin_level_binary"  "bilirubin_level"        
[57] "max..PCT"                "PCT_level_binary"       
[59] "PCT_level"

In [10]:
write.csv(merged_data, "max_numerical_correlations_with_factors.csv")

In [ ]:
# log transforming clinical values and correlating them with factor values

In [5]:
# Load required libraries
library(dplyr)
library(ggplot2)

# Assuming your data is in 'merged_data'
# If you need to load it, uncomment and modify:
# merged_data <- readRDS("path/to/your/data.rds")
# or
# load("path/to/your/data.RData")

#' Function to apply log transformation with handling for zeros and negatives
#' @param x numeric vector
#' @return log-transformed vector
log_transform <- function(x) {
  # log(x + 1) transformation
  # This handles zeros (log(0+1) = 0) and shifts all values
  return(log(x + 1))
}

#' Function to create correlation plot with log transformation
#' @param data dataframe
#' @param x_var character, name of x variable
#' @param y_var character, name of y variable
#' @param log_x logical, whether to log-transform x variable
#' @param log_y logical, whether to log-transform y variable
create_log_correlation_plot <- function(data, x_var, y_var, 
                                        log_x = FALSE, log_y = TRUE) {
  
  # Create a working dataset
  plot_data <- data %>%
    select(all_of(c(x_var, y_var))) %>%
    filter(!is.na(.data[[x_var]]) & !is.na(.data[[y_var]]))
  
  # Store original values
  plot_data$x_original <- plot_data[[x_var]]
  plot_data$y_original <- plot_data[[y_var]]
  
  # Apply log transformation
  if (log_x) {
    plot_data[[x_var]] <- log_transform(plot_data[[x_var]])
  }
  
  if (log_y) {
    plot_data[[y_var]] <- log_transform(plot_data[[y_var]])
  }
  
  # Calculate correlation on transformed data
  if (nrow(plot_data) > 2) {
    cor_test <- cor.test(plot_data[[x_var]], plot_data[[y_var]], 
                         method = "pearson")
    cor_value <- cor_test$estimate
    p_value <- cor_test$p.value
  } else {
    cor_value <- NA
    p_value <- NA
  }
  
  # Create axis labels
  x_label <- if(log_x) paste0("log(", x_var, " + 1)") else x_var
  y_label <- if(log_y) paste0("log(", y_var, " + 1)") else y_var
  
  # Create plot
  p <- ggplot(plot_data, aes(x = .data[[x_var]], y = .data[[y_var]])) +
    geom_point(alpha = 0.6, size = 2) +
    geom_smooth(method = "lm", se = TRUE, color = "blue", linewidth = 1) +
    labs(
      title = paste(y_var, "vs", x_var),
      subtitle = sprintf("R = %.2f, p = %.4f (n = %d)", 
                        cor_value, p_value, nrow(plot_data)),
      x = x_label,
      y = y_label
    ) +
    theme_classic(base_size = 12) +
    theme(
      plot.title = element_text(hjust = 0.5, face = "bold"),
      plot.subtitle = element_text(hjust = 0.5)
    )
  
  return(list(plot = p, n = nrow(plot_data), cor = cor_value, p = p_value))
}

# Get all column names starting with "max."
max_columns <- grep("^max\\.", colnames(merged_data), value = TRUE)

# Get all Factor columns
factor_columns <- grep("^Factor\\d+$", colnames(merged_data), value = TRUE)

# Create output directory
output_dir <- "/ictstr01/home/icb/bhavishya.nelakuditi/mofa_workflow/scripts/correlation_plots_log"
dir.create(output_dir, recursive = TRUE, showWarnings = FALSE)

# Loop through each max. variable and each Factor
results_summary <- data.frame()

cat("Starting log-transformed correlation plot generation...\n")
cat("Transformation: log(y + 1) for max. variables\n")
cat("No outlier removal - all data points included\n\n")

for (max_var in max_columns) {
  for (factor_var in factor_columns) {
    
    # Create plot with log transformation on y-axis (max. variables)
    # Factors are not log-transformed as they can be negative
    plot_result <- create_log_correlation_plot(
      data = merged_data,
      x_var = factor_var,
      y_var = max_var,
      log_x = FALSE,  # Don't transform Factor values (can be negative)
      log_y = TRUE    # Transform max. values (log(y + 1))
    )
    
    # Save as PDF
    pdf_filename <- paste0(output_dir, "/", 
                           gsub("\\.", "_", max_var), "_vs_", 
                           factor_var, "_log.pdf")
    
    pdf(pdf_filename, width = 8, height = 6)
    print(plot_result$plot)
    dev.off()
    
    # Store results
    results_summary <- rbind(results_summary, data.frame(
      y_variable = max_var,
      x_variable = factor_var,
      n_points = plot_result$n,
      correlation = plot_result$cor,
      p_value = plot_result$p,
      transformation = "log(y + 1)",
      pdf_file = basename(pdf_filename)
    ))
    
    cat(sprintf("✓ Saved: %s (n=%d, R=%.2f, p=%.4f)\n", 
                basename(pdf_filename), 
                plot_result$n, 
                plot_result$cor, 
                plot_result$p))
  }
}

# Save summary table
write.csv(results_summary, 
          paste0(output_dir, "/correlation_summary_log.csv"), 
          row.names = FALSE)

# Create log-transformed dataset
cat("\n=== Creating log-transformed dataset ===\n")

# Create a copy of the original data
merged_data_log <- merged_data

# Log-transform all max. columns
for (max_var in max_columns) {
  # Create new column name with _log suffix
  log_col_name <- paste0(max_var, "_log")
  
  # Apply log(x + 1) transformation
  merged_data_log[[log_col_name]] <- log_transform(merged_data_log[[max_var]])
  
  cat(sprintf("  Transformed: %s -> %s\n", max_var, log_col_name))
}

# Save the log-transformed dataset in multiple formats
# 1. RDS format (recommended for R)
saveRDS(merged_data_log, paste0(output_dir, "/merged_data_log_transformed.rds"))
cat(sprintf("\n✓ Saved: merged_data_log_transformed.rds\n"))

# 2. CSV format (for Excel/other software)
write.csv(merged_data_log, 
          paste0(output_dir, "/merged_data_log_transformed.csv"), 
          row.names = FALSE)
cat(sprintf("✓ Saved: merged_data_log_transformed.csv\n"))

# 3. Save just the log-transformed columns for easy reference
log_columns_only <- merged_data_log %>%
  select(sample_id, all_of(factor_columns), ends_with("_log"))

write.csv(log_columns_only, 
          paste0(output_dir, "/log_transformed_columns_only.csv"), 
          row.names = FALSE)
cat(sprintf("✓ Saved: log_transformed_columns_only.csv\n"))

# Print summary of transformations
cat("\n=== Transformation Summary ===\n")
cat(sprintf("Original columns: %d\n", ncol(merged_data)))
cat(sprintf("Log-transformed dataset columns: %d\n", ncol(merged_data_log)))
cat(sprintf("New log-transformed columns added: %d\n", length(max_columns)))
cat(sprintf("\nNew columns created:\n"))
for (max_var in max_columns) {
  cat(sprintf("  - %s_log\n", max_var))
}

cat("\n=== Summary ===\n")
cat(sprintf("Total plots created: %d\n", nrow(results_summary)))
cat(sprintf("All data points included (no outlier removal)\n"))
cat(sprintf("Transformation applied: log(y + 1) for max. variables\n"))
cat(sprintf("Significant correlations (p < 0.05): %d\n", 
            sum(results_summary$p_value < 0.05, na.rm = TRUE)))
cat(sprintf("\nAll files saved to: %s\n", output_dir))

# View summary
print(results_summary)

# Additional: Create a comparison plot showing top correlations
cat("\n=== Top Correlations (by absolute R value) ===\n")
top_cors <- results_summary %>%
  filter(!is.na(correlation)) %>%
  arrange(desc(abs(correlation))) %>%
  head(10)
print(top_cors[, c("y_variable", "x_variable", "n_points", "correlation", "p_value")])


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Warning message:
“package ‘ggplot2’ was built under R version 4.3.3”


Starting log-transformed correlation plot generation...
Transformation: log(y + 1) for max. variables
No outlier removal - all data points included



`geom_smooth()` using formula = 'y ~ x'


✓ Saved: max__CRP_vs_Factor1_log.pdf (n=34, R=-0.30, p=0.0799)


`geom_smooth()` using formula = 'y ~ x'


...

✓ Saved: max__PCT_vs_Factor11_log.pdf (n=27, R=0.04, p=0.8451)


`geom_smooth()` using formula = 'y ~ x'


✓ Saved: max__PCT_vs_Factor12_log.pdf (n=27, R=0.11, p=0.5806)


`geom_smooth()` using formula = 'y ~ x'


✓ Saved: max__PCT_vs_Factor13_log.pdf (n=27, R=-0.12, p=0.5430)

=== Creating log-transformed dataset ===
  Transformed: max..CRP -> max..CRP_log
  Transformed: max..IL.6 -> max..IL.6_log
  Transformed: max..leukocyte.count -> max..leukocyte.count_log
  Transformed: max..Crea -> max..Crea_log
  Transformed: max..CK -> max..CK_log
  Transformed: max..CK.MB -> max..CK.MB_log
  Transformed: max..Trop.T -> max..Trop.T_log
  Transformed: max....neutrophils -> max....neutrophils_log
  Transformed: max..lactate -> max..lactate_log
  Transformed: max..Urea -> max..Urea_log
  Transformed: max..GPT -> max..GPT_log
  Transformed: max..GOT -> max..GOT_log
  Transformed: max..Bilirubin -> max..Bilirubin_log
  Transformed: max..PCT -> max..PCT_log

✓ Saved: merged_data_log_transformed.rds
✓ Saved: merged_data_log_transformed.csv
✓ Saved: log_transformed_columns_only.csv

=== Transformation Summary ===
Original columns: 59
Log-transformed dataset columns: 73
New log-transformed columns added: 14

New